# Elazığ

## Paths

In [ ]:
from pathlib import Path

ROOT = Path("../..").resolve()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass

import numpy as np
import pandas as pd

@dataclass(frozen=True)
class ElazigConfig:

    raw_filename: str = "Solar Power Plant Production Data.xlsx"
    search_root: str = str(RAW_DIR)
    output_path: str = str(OUT_DIR / "Elazig.csv")

    latitude: float = 38.6822
    longitude: float = 39.2264
    tilt_deg: float = 20.0
    azimuth_deg: float = 180.0
    altitude_m: float = 1000.0

    source_timezone: str = "Europe/Istanbul"

    col_time: str = "Zaman"
    col_power: str = "Üretim"
    col_poa: str = "Işınım(W/㎡)"
    col_temp: str = "Ortam sıcaklığı(℃)"
    col_wind: str = "Rüzgar hızı(m/s)"

    poa_source: str = "measured"

    sensor_time_shift_hours: int = -1

    sensor_dropout_min_elev: float = 10.0

    ghi_source: str = "from_poa"

    albedo: float = 0.25

    max_poa: float = 1600.0
    temp_bounds: tuple = (-40.0, 55.0)
    wind_bounds: tuple = (0.0, 50.0)

    normalization_percentile: float = 99.0

    interpolate_limit_hours: int = 2
    climatology_window_days: int = 7

    @property
    def local_tz(self) -> str:
        return self.source_timezone

CFG = ElazigConfig()

## Load raw data

In [ ]:
def find_raw_file(cfg: ElazigConfig) -> str:
    for root, _dirs, files in os.walk(cfg.search_root):
        if cfg.raw_filename in files:
            return os.path.join(root, cfg.raw_filename)
    raise FileNotFoundError(f"'{cfg.raw_filename}' not found under {cfg.search_root}")

def load_and_audit(cfg: ElazigConfig) -> pd.DataFrame:

    path = find_raw_file(cfg)

    df = pd.read_excel(path)

    missing = [c for c in (cfg.col_time, cfg.col_power, cfg.col_poa,
                           cfg.col_temp, cfg.col_wind) if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in raw file: {missing}")

    t = pd.to_datetime(df[cfg.col_time], errors="coerce")
    n_bad = int(t.isnull().sum())
    if n_bad:
        df, t = df[t.notna()].copy(), t[t.notna()]

    df = df.set_index(pd.DatetimeIndex(t)).sort_index()
    df = df[~df.index.duplicated(keep="first")]
    return df

df_raw = load_and_audit(CFG)

## Physical cleaning

In [ ]:
def clean_and_check_units(df: pd.DataFrame, cfg: ElazigConfig) -> pd.DataFrame:

    out = pd.DataFrame(index=df.index)
    out["Power"] = pd.to_numeric(df[cfg.col_power], errors="coerce")
    out["Irradiance_POA"] = pd.to_numeric(df[cfg.col_poa], errors="coerce")
    out["Temp_Ambient"] = pd.to_numeric(df[cfg.col_temp], errors="coerce")
    out["Wind_Speed"] = pd.to_numeric(df[cfg.col_wind], errors="coerce")

    out = out.mask((out < -999) | (out > 5e6))

    out["Irradiance_POA"] = out["Irradiance_POA"].clip(lower=0).where(
        out["Irradiance_POA"] <= cfg.max_poa)
    out["Power"] = out["Power"].clip(lower=0)

    lo, hi = cfg.temp_bounds
    out["Temp_Ambient"] = out["Temp_Ambient"].where(out["Temp_Ambient"].between(lo, hi))
    lo, hi = cfg.wind_bounds
    out["Wind_Speed"] = out["Wind_Speed"].where(out["Wind_Speed"].between(lo, hi))

    return out

df_clean = clean_and_check_units(df_raw, CFG)

## Solar geometry

In [ ]:
def add_solar_geometry(df: pd.DataFrame, cfg: ElazigConfig) -> pd.DataFrame:
    from pvlib.solarposition import get_solarposition
    from pvlib.irradiance import aoi as compute_aoi

    df = df.copy()
    idx_local = df.index.tz_localize(cfg.local_tz, ambiguous="NaT", nonexistent="NaT")
    valid = idx_local.notna()
    df, idx_local = df[valid], idx_local[valid]

    solpos = get_solarposition(idx_local, cfg.latitude, cfg.longitude, altitude=cfg.altitude_m)
    df["_zenith"] = solpos["zenith"].values
    df["_elevation"] = solpos["elevation"].values
    df["_aoi"] = compute_aoi(cfg.tilt_deg, cfg.azimuth_deg,
                             solpos["zenith"].values, solpos["azimuth"].values)

    df["_utc_index"] = idx_local.tz_convert("UTC")
    return df

df_geo = add_solar_geometry(df_clean, CFG)

In [ ]:
def peak_hour(series: pd.Series, hours: np.ndarray, mask: np.ndarray) -> int:
    return int(pd.Series(series[mask].values).groupby(hours[mask]).mean().idxmax())

## Relative humidity (NASA POWER)

In [ ]:
def fetch_nasa_aligned(df: pd.DataFrame, cfg: ElazigConfig) -> pd.DataFrame:
    import requests

    idx_local = df.index
    url = (
        "https://power.larc.nasa.gov/api/temporal/hourly/point"
        "?parameters=RH2M,ALLSKY_SFC_SW_DWN,T2M&community=RE"
        f"&longitude={cfg.longitude}&latitude={cfg.latitude}"
        f"&start={idx_local.min():%Y%m%d}&end={idx_local.max():%Y%m%d}&format=JSON"
    )

    empty = pd.DataFrame({"Humidity": np.nan, "GHI_nasa": np.nan}, index=idx_local)
    try:
        res = requests.get(url, timeout=180)
        res.raise_for_status()
        p = res.json()["properties"]["parameter"]
    except Exception:
        return empty

    def to_series(d: dict) -> pd.Series:
        s = pd.Series(list(d.values()),
                      index=pd.to_datetime(list(d.keys()), format="%Y%m%d%H"))
        return s.mask(s < -900)

    rh, gh = to_series(p["RH2M"]), to_series(p["ALLSKY_SFC_SW_DWN"])
    t2m = to_series(p["T2M"])

    hours = idx_local.hour.values
    day = (df["_elevation"] > 5).values
    ref_peak = peak_hour(df["_elevation"], hours, day)

    best_shift, best_gap = 0, 99
    for sh in range(-12, 13):
        probe = pd.Series((idx_local + pd.Timedelta(hours=sh)).map(gh).values, index=idx_local)
        if probe.notna().mean() < 0.5:
            continue
        pk = peak_hour(probe, hours, day)
        gap = min(abs(pk - ref_peak), 24 - abs(pk - ref_peak))
        if gap < best_gap:
            best_shift, best_gap = sh, gap

    shifted_idx = idx_local + pd.Timedelta(hours=best_shift)
    out = pd.DataFrame({
        "Humidity": shifted_idx.map(rh),
        "GHI_nasa": shifted_idx.map(gh),
        "Temp_nasa": shifted_idx.map(t2m),
    }, index=idx_local)

    return out

_nasa = fetch_nasa_aligned(df_geo, CFG)
df_geo["Humidity"] = _nasa["Humidity"].values
df_geo["GHI_nasa"] = _nasa["GHI_nasa"].values

## Irradiance: measured POA, GHI derived with GTI-DIRINT

In [ ]:
def compute_poa_from_ghi(df: pd.DataFrame, cfg: ElazigConfig) -> pd.Series:
    from pvlib.irradiance import erbs, get_total_irradiance, get_extra_radiation
    from pvlib.solarposition import get_solarposition

    idx = df.index.tz_localize(cfg.local_tz, ambiguous="NaT", nonexistent="NaT")
    sp = get_solarposition(idx, cfg.latitude, cfg.longitude, altitude=cfg.altitude_m)
    dd = erbs(df["GHI"].values, sp["zenith"].values, idx)

    common = dict(
        surface_tilt=cfg.tilt_deg, surface_azimuth=cfg.azimuth_deg,
        solar_zenith=sp["zenith"].values, solar_azimuth=sp["azimuth"].values,
        dni=np.asarray(dd["dni"]), ghi=df["GHI"].values, dhi=np.asarray(dd["dhi"]),
    )
    poa = pd.Series(np.asarray(get_total_irradiance(
        **common, dni_extra=np.asarray(get_extra_radiation(idx)), model="perez",
    )["poa_global"], dtype=float), index=df.index)

    elev = sp["elevation"].values

    bad = poa.isnull()
    if bad.any():
        iso = np.asarray(get_total_irradiance(**common, model="isotropic")["poa_global"], dtype=float)
        poa[bad] = iso[bad.values]

    poa = poa.fillna(0.0).clip(lower=0.0)
    poa[elev < 0] = 0.0
    return poa

def prepare_measured_poa(df: pd.DataFrame, cfg: ElazigConfig) -> pd.Series:
    poa = df["Irradiance_POA"].shift(cfg.sensor_time_shift_hours)
    elev = df["_elevation"]

    poa[elev < 0] = 0.0

    dropout = (elev > cfg.sensor_dropout_min_elev) & (poa < 5)
    poa[dropout] = np.nan
    return poa

def ghi_from_poa(df: pd.DataFrame, cfg: ElazigConfig) -> pd.Series:
    from pvlib.irradiance import gti_dirint, aoi as compute_aoi
    from pvlib.solarposition import get_solarposition
    from pvlib.atmosphere import alt2pres

    idx = df.index.tz_localize(cfg.local_tz, ambiguous="NaT", nonexistent="NaT")
    sp = get_solarposition(idx, cfg.latitude, cfg.longitude, altitude=cfg.altitude_m)
    aoi = compute_aoi(cfg.tilt_deg, cfg.azimuth_deg, sp["zenith"], sp["azimuth"])

    T = df["Temp_Ambient"].to_numpy(dtype=float)
    RH = df["Humidity"].clip(1, 100).to_numpy(dtype=float)
    g = np.log(RH / 100.0) + 17.625 * T / (243.04 + T)
    temp_dew = pd.Series(243.04 * g / (17.625 - g), index=idx)

    out = gti_dirint(
        poa_global=pd.Series(df["Irradiance_POA"].to_numpy(dtype=float), index=idx),
        aoi=aoi, solar_zenith=sp["zenith"], solar_azimuth=sp["azimuth"], times=idx,
        surface_tilt=cfg.tilt_deg, surface_azimuth=cfg.azimuth_deg,
        pressure=alt2pres(cfg.altitude_m), temp_dew=temp_dew, albedo=cfg.albedo,
    )

    ghi = pd.Series(np.asarray(out["ghi"], dtype=float), index=df.index)
    ghi[sp["elevation"].values < 0] = 0.0
    return ghi.clip(lower=0.0)

df_geo["Irradiance_POA_Measured"] = df_geo["Irradiance_POA"]

if CFG.poa_source == "measured":
    df_geo["Irradiance_POA"] = prepare_measured_poa(df_geo, CFG)
    if CFG.ghi_source == "from_poa":
        df_geo["GHI"] = ghi_from_poa(df_geo, CFG)
    else:
        df_geo["GHI"] = df_geo["GHI_nasa"]
else:
    if CFG.ghi_source != "nasa":
        raise ValueError("poa_source='computed' requires ghi_source='nasa'.")
    df_geo["GHI"] = df_geo["GHI_nasa"]
    df_geo["Irradiance_POA"] = compute_poa_from_ghi(df_geo, CFG)

## Unified hourly dataset

In [ ]:
def impute_with_tracking(df: pd.DataFrame, cols: list[str], cfg: ElazigConfig) -> pd.DataFrame:
    df = df.copy()
    hours, doys = df.index.hour, df.index.dayofyear

    for col in cols:
        if col not in df.columns:
            continue
        flag = df[col].isnull()
        df[f"{col}_Imputed"] = flag
        if not flag.any():
            continue

        df[col] = df[col].interpolate(method="linear", limit=cfg.interpolate_limit_hours)

        for idx in df.index[df[col].isnull()]:
            near = (hours == idx.hour) & (np.abs(doys - idx.dayofyear) <= cfg.climatology_window_days)
            df.loc[idx, col] = df.loc[near, col].mean()

    return df

def build_unified(df: pd.DataFrame, cfg: ElazigConfig) -> pd.DataFrame:

    df = df.drop(columns=["_zenith", "_elevation", "_aoi", "_utc_index"],
                 errors="ignore").copy()
    df.index = df.index.tz_localize(None) if df.index.tz else df.index

    full = pd.date_range(df.index.min().floor("D"),
                         df.index.max().ceil("D") - pd.Timedelta(hours=1), freq="h")
    df = df.reindex(full)

    df = impute_with_tracking(
        df, ["Power", "Irradiance_POA", "GHI", "Temp_Ambient", "Wind_Speed", "Humidity"], cfg)

    out = pd.DataFrame(index=df.index)
    out["utctime"] = df.index
    out["Original_Production_W"] = df["Power"]

    p_ref = np.nanpercentile(df.loc[df["Power"] > 0, "Power"], cfg.normalization_percentile)
    out["Power_PU"] = (df["Power"] / p_ref).clip(0, 1)

    out["Irradiance_POA"] = df["Irradiance_POA"]
    if "Irradiance_POA_Measured" in df.columns:
        out["Irradiance_POA_Measured"] = df["Irradiance_POA_Measured"]
    out["GHI"] = df["GHI"]
    out["Temp_Ambient"] = df["Temp_Ambient"]
    out["Wind_Speed"] = df["Wind_Speed"]
    out["Humidity"] = df["Humidity"]

    hour, month = df.index.hour, df.index.month
    out["HOUR_SIN"] = np.sin(2 * np.pi * hour / 24.0)
    out["HOUR_COS"] = np.cos(2 * np.pi * hour / 24.0)
    out["MONTH_SIN"] = np.sin(2 * np.pi * month / 12.0)
    out["MONTH_COS"] = np.cos(2 * np.pi * month / 12.0)

    out["Latitude"], out["Longitude"], out["Tilt"] = cfg.latitude, cfg.longitude, cfg.tilt_deg

    for c in [c for c in df.columns if c.endswith("_Imputed")]:
        out[c] = df[c]

    out = out.reset_index(drop=True)
    return out

df_final = build_unified(df_geo, CFG)

## Save

In [ ]:
os.makedirs(os.path.dirname(CFG.output_path), exist_ok=True)
df_final.to_csv(CFG.output_path, index=False)